# BÀI TẬP CUỐI KỲ (LAB-END): HỆ THỐNG NHẬN DIỆN KHUÔN MẶT THỜI GIAN THỰC (FACENET & MTCNN)

---
## 📚 1. NGUYÊN LÝ HOẠT ĐỘNG & NỀN TẢNG LÝ THUYẾT

### 1.1. Thuật Toán MTCNN (Phát Hiện & Căn Chỉnh Khuôn Mặt)
MTCNN giải quyết bài toán phát hiện & căn chỉnh khuôn mặt theo chuỗi 3 mạng nơ-ron liên tiếp:
- **P-Net (Mạng đề xuất)**: Quét qua ảnh ở nhiều kích thước để tìm và đề xuất các vùng nghi ngờ có chứa khuôn mặt.
- **R-Net (Mạng tinh chỉnh)**: Lọc bỏ hầu hết các vùng nhiễu giả và tinh chỉnh lại khung Bounding Box.
- **O-Net (Mạng đầu ra)**: Xác định chính xác vị trí khuôn mặt và **5 điểm đặc trưng quan trọng** (mắt trái, mắt phải, đỉnh mũi, 2 khóe miệng).
- **Căn chỉnh (Face Alignment)**: Dựa trên 5 điểm đặc trưng, xoay và đưa khuôn mặt về góc nhìn thẳng chuẩn, cắt lấy khuôn mặt và thu nhỏ về kích thước 160x160 pixel.

### 1.2. Thuật Toán FaceNet & Vector Đặc Trưng 512 Chiều
- Mô hình **InceptionResnetV1** ánh xạ hình ảnh khuôn mặt thành một chuỗi gồm **512 số thực (Vector Embedding 512 chiều)** biểu diễn đặc trưng riêng của khuôn mặt đó.
- **Nguyên lý Triplet Loss**: Kéo 2 vector đặc trưng của cùng một người lại thật gần nhau và đẩy 2 vector của 2 người khác nhau ra thật xa nhau.
- **Chuẩn hóa Vector**: Tất cả các vector được quy đổi về cùng một độ dài chuẩn bằng 1 để so sánh công bằng.

### 1.3. Phép Đo Độ Tương Đồng & Phân Tích Chọn Ngưỡng Threshold ($0.55 - 0.70$)
- **Độ tương đồng (Cosine Similarity)**: Tính góc giữa 2 vector đặc trưng 512 chiều (trả về kết quả từ 0 tới 1). Càng tiến về 1.0 càng giống hệt nhau.
- **Phân Tích Chọn Ngưỡng Threshold**:
  - **Ngưỡng Tiêu chuẩn ($0.70$)**: Áp dụng trong điều kiện phòng thí nghiệm tiêu chuẩn (ảnh studio nét cao, nhìn thẳng trực diện, ánh sáng chuẩn).
  - **Ngưỡng Thực tế Webcam ($0.55 - 0.60$)**:
    - *Biến thiên ánh sáng & Góc nghiêng*: Ngồi trước webcam laptop, ánh sáng phòng thay đổi và góc nghiêng đầu khiến độ tương đồng của cùng 1 người tụt xuống khoảng $0.58 - 0.75$. Ngưỡng $0.70$ quá cứng nhắc dễ gây ra lỗi nhầm người quen thành `Unknown` (bỏ sót người thật).
    - *Nhiễu camera laptop*: Cảm biến webcam làm giảm độ sắc nét của đặc trưng.
    - *Độ an toàn*: Độ tương đồng giữa 2 người khác nhau luôn thấp dưới $0.40$. Ngưỡng **$0.55 - 0.60$** giúp webcam nhận diện mượt mà và phân biệt chính xác người lạ.

### 2. Import Các Thư Viện & Khởi Tạo Môi Trường

In [ ]:
import os
import sys
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import importlib

# Xác định thư mục gốc của bài Lab-cuoi
current_dir = os.getcwd()
if os.path.exists(os.path.join(current_dir, "data", "input")):
    base_dir = current_dir
elif os.path.exists(os.path.join(current_dir, "..", "data", "input")):
    base_dir = os.path.abspath(os.path.join(current_dir, ".."))
else:
    base_dir = current_dir

notebook_dir = os.path.join(base_dir, "notebook")
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

import labend
importlib.reload(labend)
from labend import FaceRecognitionSystem, imread_unicode, imwrite_unicode
print("Thư viện và module labend.py đã được import thành công!")
print(f"Base Dir: {base_dir}")
print(f"PyTorch Version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

### 3. Khởi Tạo Hệ Thống Nhận Diện Khuôn Mặt (FaceRecognitionSystem)
Hệ thống tự động khởi tạo mô hình **MTCNN** cho Face Detection và **InceptionResnetV1** (FaceNet) cho Feature Extraction.

In [ ]:
# Khởi tạo instance của hệ thống với ngưỡng threshold = 0.7
system = FaceRecognitionSystem(threshold=0.7)
print(f"Device đang sử dụng: {system.device}")
print(f"Ngưỡng Similarity mặc định: {system.threshold}")

### 4. Thử Nghiệm Phát Hiện Khuôn Mặt Bằng MTCNN (Face Detection & Landmarks)

In [ ]:
# Tên file ảnh thử nghiệm trong thư mục data/input/
image_name = "face.jpg"  # Có thể đổi thành meme.jpg hoặc memetest.jpg nếu muốn
test_img_path = os.path.join(base_dir, "data", "input", image_name)
if not os.path.exists(test_img_path):
    test_img_path = os.path.join(base_dir, "data", "input", "memetest.jpg")

img_bgr = imread_unicode(test_img_path)
if img_bgr is not None:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    
    # Phát hiện Bounding Boxes & Landmarks bằng MTCNN
    boxes, probs, landmarks = system.mtcnn.detect(pil_img, landmarks=True)
    
    num_faces = len(boxes) if (boxes is not None and len(boxes) > 0) else 0
    print(f"File ảnh: {test_img_path}")
    print(f"Số lượng khuôn mặt phát hiện được: {num_faces}")
    
    # Trực quan hóa ảnh có Bounding Box và Landmarks
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img_rgb)
    if boxes is not None and len(boxes) > 0:
        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(c) for c in box]
            prob = probs[i] if (probs is not None and i < len(probs) and probs[i] is not None) else 1.0
            prob_str = f"{prob:.2f}" if isinstance(prob, (int, float, np.floating)) else "1.00"
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 10, 0), prob_str, color='lime', fontsize=12, backgroundcolor='black')
        if landmarks is not None:
            for lm in landmarks:
                if lm is not None:
                    for pt in lm:
                        ax.plot(pt[0], pt[1], 'ro', markersize=3)
    plt.title("Kết quả Phát hiện Khuôn mặt bằng MTCNN")
    plt.axis('off')
    plt.show()
else:
    print(f"Không thể tải ảnh từ path: {test_img_path}")

### 5. Trích Xuất Vector Embedding (512D) & So Sánh Độ Tương Đồng (Cosine Similarity)
FaceNet biến đổi mỗi khuôn mặt thành vector 512 chiều. Khoảng cách giữa 2 vector càng gần thì giá trị Cosine Similarity càng tiến về 1.0.

In [ ]:
ref_img_path = test_img_path
# 1. Đăng ký ảnh khuôn mặt mẫu vào cơ sở dữ liệu hệ thống
if os.path.exists(ref_img_path):
    system.register_face("Duy", ref_img_path)

# 2. Phát hiện và trích xuất embedding từ ảnh thử nghiệm
if img_bgr is not None:
    boxes, probs, landmarks, embeddings = system.detect_and_embed(img_bgr)
    
    for i, emb in enumerate(embeddings):
        print(f"\n--- Khuôn mặt #{i+1} ---")
        print(f"Kích thước Vector Embedding: {emb.shape}")
        print(f"Chuẩn L2 Norm: {np.linalg.norm(emb):.4f}")
        
        # So sánh với mẫu đã đăng ký
        if "Duy" in system.registered_faces:
            ref_emb = system.registered_faces["Duy"]
            sim = system.compute_similarity(emb, ref_emb)
            print(f"Cosine Similarity với Duy: {sim:.4f}")
            if sim > 0.7:
                print("==> ĐÁNH GIÁ: MATCHED (Đã khớp khuôn mặt: Duy)")
            else:
                print("==> ĐÁNH GIÁ: UNKNOWN (Khuôn mặt không khớp / Chưa biết)")

### 6. Nhận Diện Khuôn Mặt Trên Ảnh Tĩnh & Xuất Kết Quả Đầu Ra

In [ ]:
out_img_path = os.path.join(base_dir, "data", "output", "result_labend.jpg")

# Chạy quy trình nhận diện hoàn chỉnh trên ảnh
annotated_bgr, results = system.recognize_image_file(test_img_path, out_img_path, threshold=0.7)

# Hiển thị ảnh gốc và ảnh nhận diện đã gán nhãn Matched/Unknown
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Ảnh Đầu Vào Gốc (Input Image)")
axes[0].axis('off')

axes[1].imshow(annotated_rgb)
axes[1].set_title("Kết Quả Nhận Diện (FaceNet + MTCNN)")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\nChi tiết kết quả nhận diện từng khuôn mặt:")
for r in results:
    print(f" - Box: {r['box']} | Status: {r['status']} | Name: {r['name']} | Similarity: {r['similarity']:.4f}")

### 7. Khởi Chạy Nhận Diện Thời Gian Thực Trên Webcam Trực Tiếp

Để mở luồng nhận diện thời gian thực qua Webcam thiết bị:
- Nhấn phím `'q'` hoặc `'ESC'` hoặc nút `[X]` góc cửa sổ để thoát.
- Nhấn phím `'s'` để chụp ảnh nhận diện hiện tại và lưu vào thư mục `data/output/`.

In [ ]:
import os
import sys
import importlib
import labend
importlib.reload(labend)
from labend import FaceRecognitionSystem

# 1. Đảm bảo khởi tạo system nếu chưa chạy các ô phía trên
if "system" not in locals():
    system = FaceRecognitionSystem(threshold=0.7)

# 2. Xác định đường dẫn ảnh mẫu face.jpg trong data/input/
current_dir = os.getcwd()
if os.path.exists(os.path.join(current_dir, "data", "input")):
    base_dir = current_dir
elif os.path.exists(os.path.join(current_dir, "..", "data", "input")):
    base_dir = os.path.abspath(os.path.join(current_dir, ".."))
else:
    base_dir = current_dir

face_path = os.path.join(base_dir, "data", "input", "face.jpg")
if not os.path.exists(face_path):
    face_path = os.path.join(base_dir, "data", "input", "memetest.jpg")

# 3. Đăng ký ảnh khuôn mặt mẫu Duy từ face.jpg nếu CSDL rỗng
if len(system.registered_faces) == 0:
    system.register_face("Duy", face_path)

# 4. MỞ LUỒNG WEBCAM TRỰC TIẾP (flip_horizontal=True lật gương selfie tự nhiên)
system.run_webcam(threshold=0.7, camera_id=0, flip_horizontal=True)